## try to combine as follow :
"""
I have these csvs, 1- screen_clusters_k80.tsv , separator is tab \t , has the following columns: screen_key app_id trace_id screen_id cluster_id 2- 2- UI metadat ui_details.csv separator is comma with the following columns UI Number App Package Name Interaction Trace Number UI Number in Trace 3- design_topics.csv with comma has only screen_id topic I want to create a new csv cmerging these 3 as follow: screen_id from file 1 is the same as "UI Number in Trace" in file 2 which is unique and to make sure from that app_id in file 1 should be identical to "App Package Name" where it can duplicate and screen_id in file 3 is the same as "UI Number" in file 2 note that all screens has clusters but not all screns hase topic id, so the final file should merge them and keep many screns without topic

In [6]:
import pandas as pd

# --------- input files ----------
CLUSTERS_TSV = "processed_data/clusters/screen_clusters_k20.tsv"   # sep = \t
UI_DETAILS   = "../../dataset/2- UI metadat ui_details.csv"            # sep = ,
TOPICS       = "../../dataset/design_topics.csv"         # sep = ,

OUT_CSV      = "merged_clusters_ui_topics_k20.csv"


In [7]:
import pandas as pd

# ---------- Load ----------
f1 = pd.read_csv(CLUSTERS_TSV, sep="\t", dtype=str)
f2 = pd.read_csv(UI_DETAILS, sep=",", dtype=str)          # your "UI metadat ui_details.csv"
f3 = pd.read_csv(TOPICS, sep=",", dtype=str)

# Normalize column names exactly as you described (adjust filenames/columns if needed)
# file1 columns: screen_key app_id trace_id screen_id cluster_id
# file2 columns: UI Number, App Package Name, Interaction Trace Number, UI Number in Trace
# file3 columns: screen_id, topic

# ---------- Step 1: Merge clusters -> metadata by in-trace screen id + app match ----------
merged_12 = f1.merge(
    f2,
    how="left",
    left_on=["screen_id", "app_id"],
    right_on=["UI Number in Trace", "App Package Name"],
    indicator=True
)

# Optional: diagnose mismatches (clustered screens that didn't find metadata)
unmatched = merged_12[merged_12["_merge"] == "left_only"]
print(f"[INFO] clustered screens with NO metadata match: {len(unmatched)}")

# ---------- Step 2: Merge topics using global unique screen id ----------
# file3.screen_id matches file2["UI Number"]
merged_all = merged_12.merge(
    f3,
    how="left",
    left_on="UI Number",
    right_on="screen_id",
    suffixes=("", "_topic")
)

# ---------- Cleanup ----------
# Keep original file1 screen_id (in-trace) AND the global UI Number, and the topic
# Drop helper columns you don't want
cols_to_drop = ["_merge", "screen_id_topic"]  # from f3 join
for c in cols_to_drop:
    if c in merged_all.columns:
        merged_all = merged_all.drop(columns=c)

# Optional: reorder columns nicely (edit as you like)
preferred = [
    "screen_key", "app_id", "trace_id", "screen_id", "cluster_id",
    "UI Number", "App Package Name", "Interaction Trace Number", "UI Number in Trace",
    "topic"
]
existing_preferred = [c for c in preferred if c in merged_all.columns]
rest = [c for c in merged_all.columns if c not in existing_preferred]
merged_all = merged_all[existing_preferred + rest]

# ---------- Save ----------
merged_all.to_csv(OUT_CSV, index=False)
print("[DONE] wrote merged_clusters_metadata_topics.csv")


[INFO] clustered screens with NO metadata match: 0
[DONE] wrote merged_clusters_metadata_topics.csv


In [ ]:
### old not working ....
# --------- load ----------
df1 = pd.read_csv(CLUSTERS_TSV, sep="\t", dtype=str)  # file 1
df2 = pd.read_csv(UI_DETAILS, sep=",", dtype=str)     # file 2
df3 = pd.read_csv(TOPICS, sep=",", dtype=str)         # file 3

# Normalize column names (keep original too, but make access robust)
df2 = df2.rename(columns={
    "UI Number": "ui_number",
    "App Package Name": "app_package_name",
    "Interaction Trace Number": "interaction_trace_number",
    "UI Number in Trace": "ui_number_in_trace",
})
df3 = df3.rename(columns={
    "screen_id": "ui_number",   # file3.screen_id == file2.UI Number
    "topic": "topic"
})

# --------- merge topics onto UI details (LEFT: keep all df2 even if no topic) ----------
df2t = df2.merge(df3[["ui_number", "topic"]], on="ui_number", how="left")

# --------- merge df1 (clusters) with df2t (UI details + topic) ----------
# file1.screen_id == file2.UI Number in Trace
merged = df1.merge(
    df2t,
    left_on="screen_id",
    right_on="ui_number_in_trace",
    how="left",              # keep ALL screens from df1 (clusters exist for all)
    suffixes=("", "_ui")
)

# --------- consistency check: app_id must match App Package Name ----------
# (App Package Name can duplicate; we just verify per-row match when df2 info exists)
merged["app_match"] = (
    merged["app_id"].fillna("") == merged["app_package_name"].fillna("")
)

# Optional: report mismatches where UI info exists but app_id differs
mismatch = merged[(merged["app_package_name"].notna()) & (~merged["app_match"])]
print(f"Total rows (clusters): {len(df1)}")
print(f"Rows with UI-details matched: {merged['app_package_name'].notna().sum()}")
print(f"Topic present: {merged['topic'].notna().sum()}")
print(f"App-ID mismatches (should be 0 ideally): {len(mismatch)}")
if len(mismatch) > 0:
    print("Example mismatches:")
    print(mismatch[["screen_id", "app_id", "app_package_name"]].head(10).to_string(index=False))

# --------- write ----------
merged.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUT_CSV}")


Total rows (clusters): 66256
Rows with UI-details matched: 5961403
Topic present: 122174
App-ID mismatches (should be 0 ideally): 5895091
Example mismatches:
screen_id                     app_id                   app_package_name
      221 B4A.BigFivePersonalityTest               yong.app.videoeditor
      221 B4A.BigFivePersonalityTest                   co.bytemark.cmta
      221 B4A.BigFivePersonalityTest          com.aminor.weatherstation
      221 B4A.BigFivePersonalityTest         com.appspot.nycsubwaytimes
      221 B4A.BigFivePersonalityTest  com.skiracer.nautical_astore_lite
      221 B4A.BigFivePersonalityTest            com.atistudios.italk.it
      221 B4A.BigFivePersonalityTest com.stillnewagain.yenininnilerimiz
      221 B4A.BigFivePersonalityTest              com.anttek.clockwiget
      221 B4A.BigFivePersonalityTest  com.love.caller.screen.sprite.coc
      221 B4A.BigFivePersonalityTest                       com.laprensa


OSError: [Errno 28] No space left on device

### add new columns image_id to tsv 

In [1]:
import pandas as pd
import os

df = pd.read_csv("processed_data/clip/clip_test.tsv", sep="\t")

df["screen_id"] = df["image_path"].apply(
    lambda p: os.path.splitext(os.path.basename(p))[0]
)

df.to_csv("processed_data/clip/output.tsv", sep="\t", index=False)


In [2]:
import pandas as pd
import os

df = pd.read_csv("processed_data/clip/clip_test.tsv", sep="\t")

# create screen_id
df["screen_id"] = df["image_path"].apply(
    lambda p: os.path.splitext(os.path.basename(p))[0]
)

# reorder columns
cols = list(df.columns)
cols.remove("screen_id")

trace_idx = cols.index("trace_id")
cols.insert(trace_idx + 1, "screen_id")

df = df[cols]

df.to_csv("processed_data/clip/output2.tsv", sep="\t", index=False)
